# 보충 연습. 보고서 구성 바꾸기 — 문서 에이전트 고급 설정
### 예제 10 · 10-8을 마친 뒤 · 2일 차 · 40분

> **실행 방법**: 위에서부터 순서대로 실행하세요. 이 노트북은 **Day2 폴더 안에** 두고 엽니다. (`exercises/custom_report.py` 파일이 함께 있어야 합니다)

09-8에서 본 것처럼 Word 보고서는 **AI가 쓴 장 + Python이 정해진 대로 쓴 부분**으로 만들어집니다.
이번에는 그 두 가지를 모두 직접 바꿉니다. 연습 네 가지를 마치면 **종합 미션**(`12_종합미션_지점장요청.ipynb`)을 풉니다.

| 연습 | 바꾸는 것 | 설정 이름 |
|---|---|---|
| 1 | AI가 쓸 **장(섹션) 종류**를 골라서 늘리기 | `rpt.SECTIONS` |
| 2 | **장 제목 · 문단** 글자 바꾸기 (`add_heading` · `add_paragraph`) | `rpt.LAYOUT[번호]` |
| 3 | **표 · 차트 종류와 순서** 바꾸기 | `rpt.LAYOUT` |
| 4 | **Python이 쓰는 부분** — 보고서 제목 · 작성자 · 유의사항 · 파일 이름 | `rpt.REPORT_TITLE` · `rpt.AUTHOR` · `rpt.CAUTIONS` |

## 0. 실습 준비

FinanceAgent를 불러오고, 이 PC의 로컬 AI(Ollama)가 켜져 있는지 확인합니다.
`AI 연결 확인: qwen3:4b-instruct 준비 완료`가 나오면 성공입니다.

`DATA`는 **실습 데이터 파일의 위치**(`data/bank_customers.csv`, 1일 차 고객 데이터 45,211건)를 담아 둔 이름입니다. 아래 셀에서 `DATA`라고 쓰면 이 파일을 뜻합니다. 다른 파일로 분석하려면 이 줄의 파일 이름만 바꿉니다.

마지막 두 줄은 보고서 설정 모듈(`custom_report`)을 `rpt`라는 짧은 이름으로 불러와 처음 상태로 맞춥니다.

In [ ]:
import json, urllib.request
from pathlib import Path
from finance_agent import FinanceAgent
import llm_config

DATA = "data/bank_customers.csv"      # 실습 데이터 파일 위치
models = [m["name"] for m in json.load(urllib.request.urlopen("http://127.0.0.1:11434/api/tags"))["models"]]
print("데이터 파일:", Path(DATA).is_file())
print("AI 연결 확인:", llm_config.OLLAMA_MODEL, "준비 완료" if llm_config.OLLAMA_MODEL in models else "→ 모델이 없습니다. 강사에게 알려 주세요")
agent = FinanceAgent(provider="ollama")

from exercises import custom_report as rpt
rpt.reset()

## 연습 0. 기본 보고서 만들고 뼈대 보기

`rpt.show_layout()`은 지금 보고서 **구성표**를, `rpt.show_docx(파일)`는 만들어진 Word의 **뼈대**(제목 · 문단 · 표 · 차트 순서)를 보여 줍니다.
`agent.run("ex11", ...)`은 예제 06~10과 같은 4단계로 실행됩니다.

| 구성 줄 | Word에 들어가는 것 | 누가 |
|---|---|---|
| `("큰제목", "1. 분석 목적")` | 장 제목 (`add_heading` 1단계) | 🐍 |
| `("작은제목", "가. 대상 고객")` | 작은 제목 (`add_heading` 2단계) | 🐍 |
| `("문단", "전체 고객 {total:,}명 …")` | 문단 (`add_paragraph`) — 자리표시 사용 가능 | 🐍 |
| `("AI", "핵심 요약")` | AI가 쓴 장 하나 (SECTIONS에 있는 이름) | 🤖 |
| `("AI모두",)` | SECTIONS 전부 (작은제목 + 내용) | 🤖 |
| `("표", "연령대")` | 고객수 · 가입자수 · 가입률 표 | 🐍 |
| `("차트", "직업군", "가입률(%)", "가로막대")` | 차트 그림 | 🐍 |
| `("유의사항",)` · `("숫자검증",)` · `("쪽나눔",)` | 유의사항 목록 · AI 숫자 검증 · 다음 쪽 | 🐍 |

- **문단 자리표시**: `{request}` 업무 요청 · `{total:,}` 전체 고객수 · `{subscribed:,}` 가입자수 · `{rate}` 가입률 · `{author}` 작성자 · `{today}` 작성일 · `{file}` 데이터 파일
- **기준**: 연령대 · 직업군 · 잔액구간 · 대출상태 · 과거캠페인 / **지표**: 가입률(%) · 고객수 / **형태**: 막대 · 가로막대 · 원(고객수만)
- 구성은 **위에서부터 적은 순서대로** Word에 들어갑니다. 순서를 바꾸려면 줄의 위치를 바꿉니다.

In [ ]:
rpt.show_layout()
request = "고객군별 정기예금 가입률 분석 보고서를 작성해 주세요."
result = agent.run("ex11", DATA, request, show=False)
print()
rpt.show_docx(result["artifacts"][0])

## 연습 1. AI가 쓸 장(섹션) 고르기 — `rpt.SECTIONS`

AI가 쓸 수 있는 장은 10가지입니다(✅ = 지금 고른 장). 목록에 없는 **새 이름**을 넣어도 됩니다(그 장은 "업무 요청에 맞춰 2~3문장"으로 써 줍니다).
기본 구성의 `("AI모두",)` 자리에 고른 장이 **고른 순서대로** 들어갑니다.

In [ ]:
rpt.show_sections()

In [ ]:
rpt.SECTIONS = ["핵심 요약", "마케팅 전략 초안", "고객 안내 문구", "지점 실행 계획"]     # ← 장 고르기 (마지막은 새로 만든 장)

prompt = rpt.make_prompt(request, rpt.run_function(DATA, request))
print(prompt[prompt.index("아래"):])        # AI에게 가는 지시 부분만 보기

In [ ]:
result = agent.run("ex11", DATA, request, show=False)
rpt.show_docx(result["artifacts"][0])

## 연습 2. 장 제목 · 문단 바꾸기 — `rpt.LAYOUT[번호]`

`show_layout()`의 **번호**로 한 줄씩 바꿉니다. `insert(번호, 줄)`은 그 자리에 새 줄을 끼워 넣습니다.
문단에는 `{total:,}` 같은 **자리표시**를 쓰면 계산된 숫자가 들어갑니다.

In [ ]:
rpt.show_layout()

In [ ]:
rpt.LAYOUT[0] = ("큰제목", "Ⅰ. 분석 배경")                                              # ← 장 제목 바꾸기
rpt.LAYOUT[1] = ("문단", "다음 달 정기예금 특판을 앞두고 고객군별 가입률을 점검함.")            # ← 문단 바꾸기
rpt.LAYOUT.insert(4, ("문단", "분석 파일: {file} · 작성자: {author} · 작성일: {today}"))      # ← 새 문단 끼워 넣기

result = agent.run("ex11", DATA, request, show=False)
rpt.show_docx(result["artifacts"][0])

## 연습 3. 표 · 차트 종류와 순서 바꾸기 — `rpt.LAYOUT` 다시 쓰기

구성 전체를 목록으로 다시 씁니다. **줄의 순서 = Word의 순서**입니다. `# ←` 줄을 바꾸거나 줄을 통째로 옮겨 보세요.
실행 전에 `rpt.check_layout()`이 기준 · 지표 · 형태 이름이 맞는지 확인해 줍니다.

In [ ]:
rpt.LAYOUT = [
    ("큰제목", "Ⅰ. 분석 배경"),
    ("문단", "전체 고객 {total:,}명 중 {subscribed:,}명이 가입하여 가입률은 {rate}%임."),
    ("큰제목", "Ⅱ. 고객군별 분석"),
    ("차트", "직업군", "가입률(%)", "가로막대"),        # ← 차트 1: 기준 · 지표 · 형태
    ("표", "직업군"),                                  # ← 표 1
    ("차트", "잔액구간", "고객수", "원"),              # ← 차트 2
    ("차트", "과거캠페인", "가입률(%)", "막대"),        # ← 차트 3
    ("큰제목", "Ⅲ. AI 분석 의견"),
    ("AI", "핵심 요약"),                               # ← AI 장은 하나씩 원하는 자리에 둘 수도 있음
    ("AI", "마케팅 전략 초안"),
    ("쪽나눔",),
    ("큰제목", "Ⅳ. 유의사항"),
    ("유의사항",),
    ("숫자검증",),
]
rpt.check_layout()

In [ ]:
result = agent.run("ex11", DATA, request, show=False)
rpt.show_docx(result["artifacts"][0])

### 연습 3-2. 틀린 구성 찾기

아래 구성에는 일부러 틀린 곳이 세 군데 있습니다. `check_layout()` 결과를 보고 고쳐 보세요. (정답: 기준 "직업군", 지표 "가입률(%)", AI 장은 SECTIONS에 있는 이름)

In [ ]:
rpt.LAYOUT = [
    ("큰제목", "Ⅰ. 분석 결과"),
    ("차트", "직업", "가입률(%)", "막대"),          # ← 틀린 곳 1
    ("표", "연령대"),
    ("차트", "연령대", "가입률", "막대"),           # ← 틀린 곳 2
    ("AI", "검토 의견"),                            # ← 틀린 곳 3 (지금 SECTIONS에 없음)
]
rpt.check_layout()

## 연습 4. Python이 쓰는 부분 바꾸기 — 제목 · 작성자 · 유의사항 · 파일 이름

보고서 맨 위 제목 · 작성자 줄 · 유의사항 · 파일 이름은 AI가 아니라 Python이 씁니다 (09-8-4와 같은 원리).
파일 이름은 자동으로 `보고서제목_작성자.docx`가 됩니다.

In [ ]:
rpt.reset()                                                     # 연습 3-2의 틀린 구성을 지우고 처음 상태로
rpt.REPORT_TITLE = "고객군별 정기예금 가입률 점검 보고서"          # ← 보고서 제목
rpt.AUTHOR = "홍길동"                                            # ← 내 이름
rpt.CAUTIONS.append("본 자료는 지점 내부 검토용이며 외부 배포를 금함.")   # ← 우리 지점 유의사항

result = agent.run("ex11", DATA, request, show=False)
print("📄", result["artifacts"][0])
rpt.show_docx(result["artifacts"][0])

## 정리 — 처음 상태로 되돌리기

미션을 시작하기 전에 꼭 실행하세요.

In [ ]:
rpt.reset()

---
### ✅ 연습에서 배운 것

| 하고 싶은 일 | 코드 |
|---|---|
| AI 장 고르기 · 새 장 만들기 | `rpt.SECTIONS = ["핵심 요약", "새 장 이름"]` |
| 구성 보기 / 확인 | `rpt.show_layout()` · `rpt.check_layout()` |
| 한 줄 바꾸기 · 끼워 넣기 | `rpt.LAYOUT[0] = ("큰제목", "…")` · `rpt.LAYOUT.insert(4, ("문단", "…"))` |
| 표 · 차트 종류와 순서 | `rpt.LAYOUT = [ ... ]` 목록 다시 쓰기 |
| 제목 · 작성자 · 유의사항 | `rpt.REPORT_TITLE` · `rpt.AUTHOR` · `rpt.CAUTIONS.append("…")` |
| 만든 Word 뼈대 보기 | `rpt.show_docx(파일)` |

▶ 이제 **`12_종합미션_지점장요청.ipynb`** 를 엽니다.